# Generate Hidden States for Eagle3 Offline Training

This notebook extracts hidden states from the target VLM model for offline Eagle3 training.

**Prerequisites:** Run `generate_responses.ipynb` first!

**Estimated time:** 8-12 hours for 100K samples (A100 80GB)
**GPU memory:** ~70GB for Qwen3-VL-30B
**Output:** ~800GB .ckpt files (or ~200GB for 4B model)

## ⚠️ GPU Requirements

**IMPORTANT:**
- Qwen3-VL-30B-A3B: **Requires A100 80GB**
- Qwen3-VL-4B: Works with A100 40GB
- Qwen3-VL-2B: Works with A100 40GB (for quick tests)

After this step, you won't need the target model anymore - only the generated .ckpt files!

## Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change to AngelSlim directory
%cd /content/AngelSlim

In [ ]:
# Install dependencies
!pip install -q transformers>=4.37.0 accelerate torch datasets

## Configuration

In [ ]:
CONFIG = {
    # Model - ВАЖНО: Qwen3-VL-30B требует A100 80GB!
    'model_name': 'Qwen/Qwen3-VL-30B-A3B',  # Requires A100 80GB
    # Для A100 40GB: 'Qwen/Qwen3-VL-4B'
    # Для быстрых тестов: 'Qwen/Qwen3-VL-2B'
    
    'model_type': 'qwen3_vl',
    'torch_dtype': 'bfloat16',
    'chat_template_type': 'qwen3_vl',
    
    # Input data (generated conversations)
    'datasets': [
        '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/data_generated.jsonl',
        '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/data_generated.jsonl',
    ],
    
    # Output directory for hidden states
    'output_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/hidden_states/train',
    
    # Processing settings
    'model_max_length': 4096,
    'num_proc': 8,
    'group_size': 5000,  # Samples per subdirectory
}

print("Configuration:")
for key, value in CONFIG.items():
    if key != 'datasets':
        print(f"  {key}: {value}")
    else:
        print(f"  datasets: {len(value)} files")

## Merge Datasets

In [ ]:
import json
from pathlib import Path

# Merge all datasets into single file
merged_path = '/content/merged_conversations.jsonl'

print("Merging datasets...")
total_samples = 0

with open(merged_path, 'w', encoding='utf-8') as outfile:
    for dataset_path in CONFIG['datasets']:
        print(f"  Reading {dataset_path}...")
        with open(dataset_path, 'r', encoding='utf-8') as infile:
            for line in infile:
                outfile.write(line)
                total_samples += 1

print(f"\n✅ Merged {total_samples} samples to {merged_path}")
CONFIG['merged_dataset'] = merged_path

## Generate Hidden States

This uses the existing `generate_hidden_for_draft_model.py` script from AngelSlim.

In [ ]:
# Get cached model path
from colab_code.utils import get_cache_manager

cache_manager = get_cache_manager()
model_cache_name = CONFIG['model_name'].replace('/', '_')

# Load from cache (should already be downloaded from generate_responses.ipynb)
model_path = cache_manager.load_from_cache(model_cache_name)

if model_path is None:
    print("Model not in cache, downloading...")
    model_path = cache_manager.download_and_cache_model(
        model_name_or_path=CONFIG['model_name'],
        cache_name=model_cache_name,
    )

print(f"✅ Model path: {model_path}")

In [ ]:
import subprocess

print("="*60)
print("Generating hidden states...")
print("This will take 8-12 hours for 100K samples")
print("="*60)

cmd = [
    'python', 'tools/generate_hidden_for_draft_model.py',
    '--modal_type', 'VLM',
    '--dataset_path', CONFIG['merged_dataset'],
    '--model_name', model_path,
    '--target_backend', 'hf',
    '--torch_dtype', CONFIG['torch_dtype'],
    '--model_max_length', str(CONFIG['model_max_length']),
    '--chat_template_type', CONFIG['chat_template_type'],
    '--outdir', CONFIG['output_dir'],
    '--num_proc', str(CONFIG['num_proc']),
    '--target_model_type', CONFIG['model_type'],
]

print("\nCommand:")
print(' '.join(cmd))
print()

result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ Hidden states generated successfully!")
else:
    print(f"\n❌ Generation failed with code {result.returncode}")

## Validation

In [ ]:
import torch
from pathlib import Path

output_dir = Path(CONFIG['output_dir'])

# Find all .ckpt files
ckpt_files = list(output_dir.rglob('*.ckpt'))

print(f"Found {len(ckpt_files)} checkpoint files")

if ckpt_files:
    # Load first checkpoint to check structure
    sample_ckpt = torch.load(ckpt_files[0])
    
    print("\nSample checkpoint structure:")
    for key, value in sample_ckpt.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {value.shape} ({value.dtype})")
        else:
            print(f"  {key}: {type(value)}")
    
    # Estimate total size
    total_size = sum(f.stat().st_size for f in ckpt_files) / (1024**3)
    print(f"\nTotal size: {total_size:.2f} GB")
    
    # Required keys for offline training
    required_keys = ['input_ids', 'target_hiddens', 'hidden_states', 'loss_mask']
    missing_keys = [k for k in required_keys if k not in sample_ckpt]
    
    if missing_keys:
        print(f"\n⚠️ Missing required keys: {missing_keys}")
    else:
        print("\n✅ All required keys present!")
else:
    print("\n❌ No checkpoint files found!")

## Summary

In [ ]:
from colab_code.utils import get_cache_manager

cache_manager = get_cache_manager()
storage = cache_manager.estimate_storage(
    include_models=True,
    include_datasets=True,
    include_checkpoints=False,
    include_hidden_states=True,
)

print("="*60)
print("HIDDEN STATES GENERATION COMPLETE")
print("="*60)

print(f"\n📊 Statistics:")
print(f"  Total samples: {len(ckpt_files)}")
print(f"  Hidden states size: {storage['hidden_states']:.2f} GB")
print(f"  Total Drive usage: {storage['total']:.2f} GB")

print(f"\n📁 Saved to:")
print(f"  {CONFIG['output_dir']}")

print(f"\n✅ Next step:")
print(f"  Run eagle3_qwen3vl_training_offline.ipynb to train draft model")
print(f"  (You can now unload the target model - only draft training remains!)")